# 🚀 Bước 3: Huấn luyện PhoBERT (4 Epochs) trên Google Colab / Kaggle

Notebook này hoàn toàn **tự chứa (self-contained)** để chạy trên Google Colab (hoặc Kaggle) có GPU:
1. Kiểm tra GPU và kết nối Google Drive (để lưu checkpoint chống mất dữ liệu khi ngắt kết nối).
2. Clone dự án từ GitHub (`https://github.com/dtc245160044-code/final-SIC.git`, nhánh `feature/models-training`).
3. Cài đặt đúng phiên bản các thư viện cần thiết.
4. Chạy thử chế độ `--smoke` (300 dòng, 1 epoch) để kiểm tra lỗi trước khi chạy thật.
5. Huấn luyện thật **4 epochs** trên tập Train (23.880 dòng), đánh giá trên tập Val (2.985 dòng) để so sánh với mốc Baseline (`F1-macro Val = 0.6819`).
6. Xuất xác suất dự đoán trên tập Test (`phobert_test_predictions.csv`) **nhưng tuyệt đối chưa chấm điểm Test** (để dành chấm đúng 1 lần ở Bước 4).
7. Đóng gói toàn bộ kết quả (`models/phobert/best/`, các file CSV, biểu đồ và log huấn luyện) thành 1 file `phobert_ket_qua_buoc3.zip` để tải về giải nén thẳng vào `D:\final SIC`.

## 1️⃣ Bật GPU và kết nối Google Drive
- Trước khi chạy ô dưới, hãy kiểm tra trên thanh menu Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU**.
- Ô này cũng kết nối Google Drive để lưu checkpoint tự động sau mỗi epoch.

In [ ]:
import os
import torch

# 1. Kiểm tra GPU
if torch.cuda.is_available():
    print(f"✅ Đã bật GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    raise RuntimeError("❌ CHƯA BẬT GPU! Hãy vào menu Runtime -> Change runtime type -> chọn T4 GPU rồi chạy lại.")

# 2. Kết nối Google Drive để lưu checkpoint an toàn (nếu chạy trên Colab)
USE_GOOGLE_DRIVE = True
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/final_SIC_phobert_backup"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print(f"✅ Đã kết nối Google Drive. Thư mục sao lưu: {DRIVE_BACKUP_DIR}")
    except Exception as e:
        print(f"ℹ️ Không chạy trên Google Colab hoặc bỏ qua Drive ({e}). Sẽ lưu cục bộ.")
        USE_GOOGLE_DRIVE = False

## 2️⃣ Clone repo dự án và kiểm tra dữ liệu `data/processed/`
- Mặc định lệnh dưới sẽ clone nhánh `feature/models-training` từ `https://github.com/dtc245160044-code/final-SIC.git`.
- **Nếu repo đang để chế độ Private:** bạn chỉ cần đổi `IS_PRIVATE_REPO = True`, ô nhập ẩn `getpass` sẽ hiện ra để bạn dán **GitHub Personal Access Token (PAT)** (vào *GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token*, tích quyền `repo`). Token nhập qua `getpass` không bị lưu vào file hay hiện trên màn hình.
- Ba file `03_train.csv`, `03_val.csv`, `03_test.csv` đã có sẵn trong repo trên nhánh `feature/models-training`.

In [ ]:
import os
import getpass
import subprocess

REPO_URL = "https://github.com/dtc245160044-code/final-SIC.git"
BRANCH = "feature/models-training"
WORK_DIR = "/content/final-SIC"
IS_PRIVATE_REPO = False  # Đổi thành True nếu repo của bạn để Private

if not os.path.exists(WORK_DIR):
    if IS_PRIVATE_REPO:
        token = getpass.getpass("Nhập GitHub Personal Access Token (ẩn): ").strip()
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "-b", BRANCH, auth_url, WORK_DIR], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORK_DIR], check=True)

%cd /content/final-SIC
!git pull origin {BRANCH}

# Kiểm tra 3 file dữ liệu đã chia trong data/processed/
for fname in ["03_train.csv", "03_val.csv", "03_test.csv"]:
    fpath = os.path.join("data/processed", fname)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"✅ Đã có {fpath} ({size_mb:.2f} MB)")
    else:
        print(f"⚠️ Thiếu {fpath}! Hãy tải file từ máy lên thư mục data/processed/.")

## 3️⃣ Cài đặt thư viện đúng phiên bản
Cài đặt các thư viện cần thiết cho PhoBERT (`transformers`, `datasets`, `accelerate`, `pyvi`, `scikit-learn`, `seaborn`) và kiểm tra phiên bản.

In [ ]:
!pip install -q "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.29.0" "pyvi==0.1.1" "scikit-learn>=1.4.0" "seaborn>=0.13.0" certifi

import transformers, datasets, accelerate, sklearn, torch
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"scikit-learn : {sklearn.__version__}")

## 4️⃣ Chạy thử chế độ `--smoke` để kiểm tra trước khi huấn luyện thật
- Lệnh này chỉ lấy 300 dòng train, 100 dòng val, 100 dòng test và chạy 1 epoch (mất khoảng 15–20 giây trên GPU T4).
- Nếu bước này báo hoàn tất không lỗi, bạn hoàn toàn yên tâm chạy bước huấn luyện thật bên dưới.

In [ ]:
import shutil

!python scripts/run_phobert.py --smoke

# Dọn thư mục tạm sau khi kiểm tra smoke thành công
shutil.rmtree("models/phobert_smoke", ignore_errors=True)
shutil.rmtree("outputs/smoke", ignore_errors=True)
print("✅ Kiểm tra --smoke thành công và đã dọn dẹp thư mục tạm!")

## 5️⃣ Huấn luyện thật PhoBERT (4 Epochs) + Lưu checkpoint vào Google Drive
- **Mặc định:** `BATCH_SIZE = 16`, `GRAD_ACCUM = 2` (tương đương batch size hiệu dụng = 32), `EPOCHS = 4`, bật `fp16` tự động trên GPU.
- **Xử lý khi hết VRAM (`CUDA out of memory`):** sửa `BATCH_SIZE = 8` và `GRAD_ACCUM = 4` (hoặc `BATCH_SIZE = 4` và `GRAD_ACCUM = 8`) rồi chạy lại ô này.
- **Xử lý khi Colab ngắt kết nối giữa chừng:** vì checkpoint mỗi epoch được lưu trực tiếp trên Google Drive (`DRIVE_BACKUP_DIR/checkpoints`), bạn chỉ cần bật `RESUME_IF_INTERRUPTED = True` rồi chạy lại ô này để học tiếp từ epoch dở dang.

In [ ]:
import os
import shutil

# ===================== CẤU HÌNH HUẤN LUYỆN =====================
EPOCHS = 4
BATCH_SIZE = 16              # Nếu lỗi CUDA Out of Memory -> giảm xuống 8
GRAD_ACCUM = 2               # Nếu giảm BATCH_SIZE xuống 8 -> tăng lên 4
RESUME_IF_INTERRUPTED = True # Tự động nối tiếp từ checkpoint gần nhất nếu có
# ===============================================================

# Nếu có Google Drive thì lưu quá trình train trực tiếp lên Drive để không mất khi rớt mạng
ckpt_dir = f"{DRIVE_BACKUP_DIR}/models_phobert" if USE_GOOGLE_DRIVE else "models/phobert"
resume_flag = "--resume" if RESUME_IF_INTERRUPTED else ""

cmd = f"python scripts/run_phobert.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --grad_accum {GRAD_ACCUM} --model_dir \"{ckpt_dir}\" --out_dir outputs {resume_flag}"
print("Đang chạy lệnh:", cmd)
!{cmd}

# Nếu lưu checkpoint trên Drive, đồng bộ thư mục best về lại models/phobert/best để đóng gói
if USE_GOOGLE_DRIVE and os.path.exists(f"{ckpt_dir}/best"):
    os.makedirs("models/phobert", exist_ok=True)
    if os.path.exists("models/phobert/best"):
        shutil.rmtree("models/phobert/best")
    shutil.copytree(f"{ckpt_dir}/best", "models/phobert/best")
    print("✅ Đã đồng bộ models/phobert/best từ Google Drive về thư mục dự án!")

## 6️⃣ Xem Log huấn luyện qua 4 Epoch & So sánh trên tập Validation (Chưa chấm Test)
- Xem bảng `outputs/phobert_training_log.csv` (Loss và F1-macro trên tập Val sau mỗi epoch).
- Hiển thị Ma trận nhầm lẫn của PhoBERT trên tập Validation (`outputs/confusion_matrix_phobert_val.png`).
- So sánh trực tiếp **F1-macro (Val)** của PhoBERT với mốc Baseline tốt nhất ở Bước 2 (`0.6819`).

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

print("=== 1. BẢNG LOG HUẤN LUYỆN THEO EPOCH ===")
df_log = pd.read_csv("outputs/phobert_training_log.csv")
display(df_log)

print("\n=== 2. THÔNG TIN TỔNG HỢP & SO SÁNH VỚI BASELINE TRÊN TẬP VAL ===")
with open("outputs/phobert_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)
for k, v in meta.items():
    print(f"- {k}: {v}")

print("\n=== 3. MA TRẬN NHẦM LẪN TRÊN TẬP VAL ===")
display(Image(filename="outputs/confusion_matrix_phobert_val.png", width=520))

## 7️⃣ Đóng gói toàn bộ file kết quả thành `phobert_ket_qua_buoc3.zip` để tải về
Ô này kiểm tra sự tồn tại của tất cả các file cần thiết và nén lại thành 1 file zip duy nhất (giữ nguyên đường dẫn tương đối `models/phobert/best/...` và `outputs/...`).

Khi tải `phobert_ket_qua_buoc3.zip` về máy cá nhân, bạn chỉ cần **giải nén thẳng vào thư mục gốc `D:\final SIC`**.

In [ ]:
import os
import zipfile
import shutil

required_files = [
    "outputs/phobert_val_predictions.csv",
    "outputs/phobert_test_predictions.csv",
    "outputs/confusion_matrix_phobert_val.png",
    "outputs/phobert_training_log.csv",
    "outputs/phobert_training_log.json",
    "outputs/phobert_meta.json",
]

assert os.path.isdir("models/phobert/best"), "❌ Chưa thấy thư mục models/phobert/best!"
for fpath in required_files:
    assert os.path.exists(fpath), f"❌ Thiếu file: {fpath}"

zip_name = "phobert_ket_qua_buoc3.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    # 1. Nén thư mục models/phobert/best/
    for root, _, files in os.walk("models/phobert/best"):
        for file in files:
            full_path = os.path.join(root, file)
            zf.write(full_path, arcname=full_path)
    # 2. Nén các file kết quả trong outputs/
    for fpath in required_files:
        zf.write(fpath, arcname=fpath)

zip_mb = os.path.getsize(zip_name) / (1024 * 1024)
print(f"✅ Đã đóng gói thành công: {zip_name} ({zip_mb:.1f} MB)")

# Sao lưu thêm 1 bản zip vào Google Drive để tải từ Drive cho nhanh
if USE_GOOGLE_DRIVE:
    drive_zip = os.path.join(DRIVE_BACKUP_DIR, zip_name)
    shutil.copy2(zip_name, drive_zip)
    print(f"✅ Đã sao lưu file zip lên Google Drive tại: {drive_zip}")

# Tự động tải về trình duyệt (nếu chạy trên Colab)
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print("👉 Bạn có thể tải file phobert_ket_qua_buoc3.zip ở khung Files bên trái hoặc trong Google Drive.")